In [ ]:
"""
Single dataset batch metrics computation script for reproducibility.

This script computes batch-related quality metrics for a single method-dataset pair.
It can be easily configured to work with different datasets by modifying the 
load_method_output function or dataset parameters.
"""

import scanpy as sc
import pandas as pd
import os
import sys
import scib


def load_method_output(method, dataset):
    """
    Load the method output for a given method-dataset pair.
    
    This function should be implemented to load the results from your specific pipeline.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        
    Returns:
        adata: AnnData object with 'latent' representation in obsm and 'cell_type', 'batch' in obs
    """
    # TODO: Implement this function to load your results
    # Example implementation:
    # return sc.read_h5ad(f'./results/{method}/{dataset}_latent.h5ad')
    raise NotImplementedError("Please implement load_method_output() for your data loading")


def _batch_metrics(method, dataset):
    """
    Compute batch metrics for a single method-dataset pair.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        
    Returns:
        dict: Dictionary containing computed metrics
        
    Raises:
        ValueError: If required fields are missing from the AnnData object
    """
    adata = load_method_output(method, dataset)

    if adata.obsm.get('latent') is None:
        raise ValueError("Latent representation not found in adata.obsm['latent']")
    if "cell_type" not in adata.obs.columns:
        raise ValueError("Cell type labels not found in adata.obs['cell_type']")
    if "batch" not in adata.obs.columns:
        raise ValueError("Batch labels not found in adata.obs['batch']")
    
    sc.pp.neighbors(adata, use_rep="latent")
    gc_val = scib.me.graph_connectivity(adata, label_key="cell_type")
    asw_batch_val = scib.me.silhouette_batch(adata, batch_key="batch", label_key="cell_type", embed="latent")

    results = {
        "ASW-Batch": [asw_batch_val],
        "Graph Connectivity": [gc_val]
    }
    return results


def compute_clustering_metrics(method, dataset, output_dir='./results'):
    """
    Compute and save clustering metrics for a single dataset.
    
    Args:
        method (str): Name of the method
        dataset (str): Name of the dataset
        output_dir (str): Directory to save results
        
    Returns:
        pd.DataFrame: DataFrame containing the computed metrics
    """
    try:
        results_path = f'{output_dir}/{method}/{dataset}_batch_metrics.csv'
        
        if os.path.exists(results_path):
            print(f"Results already exist at {results_path}. Skipping calculation.")
            return pd.read_csv(results_path)

        os.makedirs(f'{output_dir}/{method}/', exist_ok=True)
        
        print(f"Computing metrics for {method} - {dataset}...")
        results = _batch_metrics(method, dataset)
        
        results_df = pd.DataFrame(results)
        results_df.to_csv(results_path, index=False)
        
        print(f"Results saved to {results_path}")
        print(results_df)
        
        return results_df
        
    except Exception as e:
        print(f"Error processing {method} - {dataset}: {str(e)}")
        raise



In [ ]:
METHOD = "your_method_name"  # Change this to your method name
DATASET = "your_dataset_name"  # Change this to your dataset name
OUTPUT_DIR = "./results"  # Change this to your desired output directory

# Compute metrics
metrics = compute_clustering_metrics(METHOD, DATASET, output_dir=OUTPUT_DIR)